# ตรวจไฟล์นิยาย .txt ก่อนใช้งาน

อัปโหลดไฟล์ .txt แล้วกด Run all ระบบจะบอกว่าไฟล์ครบหรือไม่

In [ ]:
import re
from google.colab import files

EXPECTED = 482
MIN_CHARS = 200
SIDEBAR = re.compile(r'(热门小说|人气小说榜|最新入库|请记住本书首发|本章未完|点击下一页|加入书架|推荐阅读)')
HEADER = re.compile(r'^第(\d+)章[^\n]*$', re.M)

def count_cn(s):
    return sum(1 for c in s if '\u4e00' <= c <= '\u9fff')

uploaded = files.upload()
path = next(iter(uploaded))
raw = uploaded[path]
try:
    text = raw.decode('utf-8')
except UnicodeDecodeError:
    text = raw.decode('gbk', errors='replace')

matches = list(HEADER.finditer(text))
if not matches:
    print('ไม่พบหัวตอนรูปแบบ 第N章 เลย — ไฟล์นี้ใช้ไม่ได้')
else:
    chapters = []
    for i, m in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chapters.append((int(m.group(1)), m.group(0), text[m.end():end]))

    nums = [c[0] for c in chapters]
    missing = sorted(set(range(1, max(nums) + 1)) - set(nums))
    dups = sorted({n for n in nums if nums.count(n) > 1})
    empty = [n for n, h, body in chapters if count_cn(body) < MIN_CHARS]
    sidebar = [n for n, h, body in chapters if SIDEBAR.search(body)]
    mismatch = []
    for n, h, body in chapters:
        inner = re.findall(r'第(\d+)章', h)
        if len(inner) >= 2 and int(inner[1]) != n:
            mismatch.append(n)

    print(f'ไฟล์: {path}  ({len(raw) / 1024:.1f} KB)')
    print(f'จำนวนตอน: {len(chapters)}  (คาดหวัง {EXPECTED})')
    print(f'เลขตอน: {min(nums)} ถึง {max(nums)}')
    print(f'ตอนที่หายไป: {len(missing)}  {missing[:20]}')
    print(f'ตอนที่ซ้ำ: {len(dups)}  {dups[:20]}')
    print(f'ตอนว่าง/สั้นผิดปกติ: {len(empty)}  {empty[:20]}')
    print(f'ตอนที่มีรายการแนะนำปน: {len(sidebar)}  {sidebar[:20]}')
    print(f'หัวตอนไม่ตรงกับเลขในชื่อ: {len(mismatch)}  {mismatch[:20]}')

    ok = (len(chapters) >= EXPECTED and not missing and not dups
          and not empty and not sidebar and not mismatch)
    print('\nผล:', 'ผ่าน — ไฟล์น่าจะครบ' if ok else 'ไม่ผ่าน — ดูรายการด้านบน')
